# 5. MSK-IMPACT: rare cancer knowledge from a clinical cohort

The MSK-IMPACT Clinical Sequencing Cohort (Zehir et al., Nat Med 2017; 10,945 tumors) mapped onto Anveshar's rare cancer catalog, with real per-type driver frequencies. Accessed via cBioPortal.

In [ ]:
import os, sys
def _find_repo():
    p = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(p, "anveshar", "__init__.py")):
            return p
        p = os.path.dirname(p)
    return None
REPO = _find_repo()
if REPO and REPO not in sys.path:
    sys.path.insert(0, REPO)
import anveshar
print("anveshar loaded from", os.path.dirname(anveshar.__file__))

In [ ]:
import requests, re, pandas as pd
from collections import defaultdict
API = "https://www.cbioportal.org/api"; STUDY = "msk_impact_2017"
def norm(s):
    toks = re.sub(r"[^a-z0-9]+", " ", (s or "").lower()).split()
    return " ".join(sorted(t for t in toks if t not in {"the","of","a","an","and","type","tumor","tumour","neoplasm","cancer","carcinoma"}))
cd = requests.get(f"{API}/studies/{STUDY}/clinical-data", params={"attributeId": "CANCER_TYPE_DETAILED",
   "clinicalDataType": "SAMPLE", "projection": "SUMMARY", "pageSize": 300000}, timeout=90).json()
type_samples = defaultdict(list)
for r in cd: type_samples[r["value"]].append(r["sampleId"])
print("tumors:", len(cd), "| detailed cancer types:", len(type_samples))

## Map MSK-IMPACT cancer types onto Anveshar's rare cancer catalog

In [ ]:
from anveshar import analysis
cat = analysis.load_catalog(); cat_norm = {}
for r in cat:
    cat_norm[norm(r["name"])] = r["name"]
    for a in r.get("aliases", []) or []: cat_norm.setdefault(norm(a), r["name"])
rows = [{"msk_type": t, "n": len(s), "rare_cancer": cat_norm.get(norm(t), "")} for t, s in type_samples.items()]
mp = pd.DataFrame(rows); matched = mp[mp.rare_cancer != ""]
print("rare cancer types:", matched.rare_cancer.nunique(), "| tumors:", int(matched.n.sum()))
matched.groupby("rare_cancer").n.sum().sort_values(ascending=False).head(15)

## Real driver frequencies for a flagship rare cancer (uveal melanoma)

In [ ]:
t = "Uveal Melanoma"; sids = type_samples.get(t, [])
muts = requests.post(f"{API}/molecular-profiles/{STUDY}_mutations/mutations/fetch",
   json={"sampleIds": sids}, params={"projection": "DETAILED"}, timeout=90).json()
gs = defaultdict(set)
for m in muts:
    g = (m.get("gene") or {}).get("hugoGeneSymbol")
    if g: gs[g].add(m["sampleId"])
pd.Series({g: round(100*len(s)/len(sids), 1) for g, s in gs.items()}).sort_values(ascending=False).head(10)

The MSK-IMPACT driver frequencies (GNAQ, GNA11, BAP1, SF3B1) reproduce known uveal melanoma biology in an independent clinical cohort, with BAP1 enriched relative to TCGA (metastatic bias).

---
*Disclaimer: this notebook produces research and educational analysis of public data, not medical advice. Confidence and validation scores summarize evidence strength, not the probability of benefit for any individual. Every clinical decision must be made by a qualified health care provider, ideally within a clinical trial.*

*Developed by Dig Vijay Kumar Yarlagadda, [digvijayky.com](https://digvijayky.com).*